# 🌳 Refauna SLM — Gemma 4 E2B com fine-tuning (QLoRA) + RAG

**Palestra:** *Pequenos modelos, conhecimento especializado: fine-tuning e RAG com os dados do Refauna* — TDC Stage @ Rec'n'Play 2026

Este notebook faz o projeto inteiro de ponta a ponta no **Google Colab Pro (L4 ou A100)**:

| Seção | O que acontece |
|---|---|
| 0 | Setup, segredos e configuração |
| 1 | Ingestão dos PDFs do Refauna → Markdown limpo |
| 2 | Chunking por seção (~400 tokens) |
| 3 | **RAG**: embeddings (EmbeddingGemma) + ChromaDB + BM25 → busca híbrida |
| 4 | Dataset sintético em PT-BR no formato RAG (estilo RAFT) |
| 5 | Curadoria, revisão humana e split por documento |
| 6 | **Fine-tuning** do Gemma 4 E2B com QLoRA (Unsloth) |
| 7 | Avaliação em 4 configurações: base, base+RAG, FT, FT+RAG |
| 8 | Exportação (LoRA, merged, GGUF) e publicação no Hugging Face |
| 9 | Levando para o MacBook M1 (Ollama) |

### Antes de rodar
1. **Ambiente de execução → Alterar tipo → GPU L4** (ou A100).
2. **Secrets do Colab** (ícone 🔑 à esquerda): `HF_TOKEN` (com permissão *write*) e `ANTHROPIC_API_KEY`.
3. No Hugging Face, aceite os termos de [`google/embeddinggemma-300m`](https://huggingface.co/google/embeddinggemma-300m) (modelo *gated*).
4. Estrutura esperada no Google Drive (o notebook procura a pasta `training-files` sozinho se o caminho for outro):
   ```
   MyDrive/refauna-slm/
   ├── training-files/
   │   ├── REF-L1-001_....pdf ...         # PDFs
   │   ├── search_results.xlsx            # sua planilha de levantamento
   │   └── sources.csv                    # MANIFESTO: o que entra, recorte de páginas, metadados
   └── gold_questions.jsonl               # opcional: {"question", "reference_answer", "doc_id"} escritas à mão
   ```
   Tudo o que o notebook gera (markdown, chunks, dataset, índice, resultados) vai para `refauna-slm/data/` e `refauna-slm/outputs/`.

> 💡 **Ideia central:** o fine-tuning ensina o modelo a *ler, filtrar e citar* contexto em PT-BR (comportamento); o RAG fornece os *fatos* rastreáveis. Por isso treinamos no mesmo formato de prompt que o RAG usa na inferência.

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q unsloth
!pip install -q pymupdf4llm chromadb rank-bm25 "sentence-transformers>=5.0" anthropic

# Verificação: todas as linhas devem mostrar ✅. Se houver ❌, rode esta célula de novo e leia o erro do pip acima.
import importlib.util
faltando = [m for m in ["unsloth", "pymupdf", "pymupdf4llm", "chromadb", "rank_bm25", "sentence_transformers", "anthropic"]
            if importlib.util.find_spec(m) is None]
print("✅ todas as bibliotecas instaladas" if not faltando else f"❌ faltando: {faltando}")

> Se o Colab pedir para **reiniciar a sessão** após a instalação, reinicie e continue a partir da célula abaixo (não precisa instalar de novo). Se você usar *Desconectar e excluir ambiente de execução*, aí sim é preciso rodar a instalação outra vez.

In [ ]:
import unsloth  # importe a Unsloth antes de transformers/trl para ativar as otimizações
import os, re, json, random, time, shutil, unicodedata, csv
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch
from google.colab import drive, userdata

drive.mount("/content/drive")

# ======================= CONFIGURAÇÃO =======================
HF_USERNAME  = "SEU_USUARIO_HF"     # <- altere
HF_PRIVATE   = True                 # publica privado; revise e depois torne público no HF

PROJECT_DIR  = Path("/content/drive/MyDrive/refauna-slm")   # <- ajuste se a pasta estiver em outro lugar do Drive
if not (PROJECT_DIR / "training-files").exists():            # tenta achar a pasta automaticamente
    found = sorted(Path("/content/drive/MyDrive").glob("**/training-files"))
    assert found, ("Pasta 'training-files' não encontrada no Meu Drive. Ajuste PROJECT_DIR. "
                   "Se a pasta foi compartilhada com você, use 'Adicionar atalho ao Drive' antes.")
    PROJECT_DIR = found[0].parent
PDF_DIR      = PROJECT_DIR / "training-files"
SOURCES_CSV  = PDF_DIR / "sources.csv"                  # manifesto: quais PDFs entram, páginas, metadados
GOLD_FILE    = PROJECT_DIR / "gold_questions.jsonl"     # opcional
DATA_DIR     = PROJECT_DIR / "data"
MD_DIR       = DATA_DIR / "markdown"
OUT_DIR      = PROJECT_DIR / "outputs"
LOCAL_DIR    = Path("/content/work")                  # disco local do Colab (rápido)
for d in (DATA_DIR, MD_DIR, OUT_DIR, LOCAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

BASE_MODEL   = "unsloth/gemma-4-E2B-it"
EMBED_MODEL  = "google/embeddinggemma-300m"
GEN_MODEL    = "claude-sonnet-5-5"   # gera o dataset sintético e atua como juiz na avaliação

LOAD_IN_4BIT = True    # True = QLoRA (4-bit) | False = LoRA 16-bit  -> rode as duas e compare!
MAX_SEQ_LEN  = 4096
LORA_R       = 16
EPOCHS       = 2
LR           = 2e-4

CHUNK_TOKENS, CHUNK_OVERLAP, MIN_CHUNK_TOKENS = 400, 60, 60
TOP_K               = 4      # trechos no contexto (treino e inferência usam o MESMO valor)
QUESTIONS_PER_CHUNK = 3      # corpus pequeno (~22 documentos) -> 3 perguntas por trecho
MAX_CHUNKS_FOR_GEN  = None   # None = todos os trechos elegíveis; use 30 para um teste barato
SEED                = 3407
RUN_NAME = f"gemma4-e2b-refauna-{'qlora' if LOAD_IN_4BIT else 'lora16'}"

REPO_LORA   = f"{HF_USERNAME}/gemma-4-e2b-refauna-lora"
REPO_MERGED = f"{HF_USERNAME}/gemma-4-e2b-refauna"
REPO_GGUF   = f"{HF_USERNAME}/gemma-4-e2b-refauna-GGUF"
REPO_SFT    = f"{HF_USERNAME}/refauna-sft-ptbr"
REPO_CHUNKS = f"{HF_USERNAME}/refauna-rag-chunks"

random.seed(SEED)
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
try:
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    ANTHROPIC_API_KEY = None
    print("⚠️ ANTHROPIC_API_KEY ausente: geração sintética e juiz LLM não vão rodar.")

from huggingface_hub import login
login(token=HF_TOKEN)
print("GPU:", torch.cuda.get_device_name(0), "| PDFs encontrados:", len(list(PDF_DIR.glob("*.pdf"))))

In [ ]:
def write_jsonl(path, rows):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

### Contrato de prompt (idêntico no treino, no Colab, no Mac e no Space)
Divergência entre o prompt de treino e o de inferência é a causa nº 1 de um modelo fine-tunado "piorar" depois de exportado. Este bloco é a fonte da verdade — o `config/prompts.yaml` do repositório deve ter exatamente o mesmo texto.

In [ ]:
SYSTEM_PROMPT = (
    "Você é o Assistente Refauna, especialista em defaunação, refaunação, reintrodução de fauna, "
    "dispersão de sementes, interações ecológicas e conservação da Mata Atlântica. "
    "Responda sempre em português do Brasil, de forma clara, didática e precisa. "
    "Quando houver CONTEXTO, use somente as informações dele e cite os trechos usados no formato [n]. "
    "Se o CONTEXTO não contiver a resposta, diga explicitamente que não encontrou essa informação "
    "nos documentos do Refauna e não invente dados, números, datas ou nomes de espécies."
)

ABSTAIN_ANSWERS = [
    "Não encontrei essa informação nos documentos do Refauna fornecidos no contexto. Os trechos disponíveis tratam de outros aspectos e não permitem responder com segurança.",
    "Os trechos recuperados não contêm a resposta para essa pergunta. Prefiro não especular: não encontrei essa informação nos documentos do Refauna.",
    "Com base no contexto fornecido, não é possível responder. Não encontrei nos documentos do Refauna dados sobre esse ponto específico.",
    "Não encontrei essa informação nos documentos do Refauna. O contexto disponível aborda temas relacionados, mas não traz esse dado.",
    "Essa informação não aparece nos trechos dos documentos do Refauna que recebi, então não consigo responder sem correr o risco de inventar.",
]
ABSTAIN_RE = re.compile(r"não (encontrei|consigo responder|é possível responder|há informaç|aparece)|não contêm a resposta", re.I)

def is_abstention(text):
    return bool(ABSTAIN_RE.search(text or ""))

NAME_SUFFIX = {"jr", "júnior", "junior", "filho", "neto", "sobrinho"}

def short_ref(c):
    """'(Cid et al., 2014 — Título curto)'. Aceita autores separados por ';' (planilha) ou ','."""
    raw = (c.get("authors") or "").strip()
    has_etal = bool(re.search(r"\bet al\b", raw))
    raw = re.sub(r"\bet al\.?", "", raw).strip(" ;,")
    parts = [a.strip() for a in re.split(r";" if ";" in raw else r",| & | and ", raw) if a.strip()]
    first = parts[0].split(",")[0] if parts else ""          # "Fernandez, F." -> "Fernandez"
    words = [w for w in first.split() if w.rstrip(".").lower() not in NAME_SUFFIX]
    surname = words[-1] if words else ""
    year = c.get("year") or "s.d."
    title = (c.get("title") or c.get("doc_id", "")).strip()
    title = title if len(title) <= 60 else title[:57].rstrip() + "..."
    who = (f"{surname} et al., " if (len(parts) > 1 or has_etal) else f"{surname}, ") if surname else ""
    return f"({who}{year} — {title})"

def format_context(chunks):
    return "\n".join(f"[{i}] {short_ref(c)} {c['text'].strip()}" for i, c in enumerate(chunks, 1))

def build_user_message(question, chunks=None):
    if chunks:
        return f"CONTEXTO:\n{format_context(chunks)}\n\nPERGUNTA: {question.strip()}"
    return question.strip()

def build_messages(question, chunks=None):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_message(question, chunks)}]

PT_SW = set("de que não para com uma os as dos das em é são também pela pelo mais como sua seu ao".split())
EN_SW = set("the and of to is are with for that this which were from by their".split())
def is_ptbr(text):
    toks = re.findall(r"\w+", (text or "").lower())
    return sum(t in PT_SW for t in toks) >= sum(t in EN_SW for t in toks)

## 1. Ingestão dos PDFs
`pymupdf4llm` converte cada página em Markdown. Depois limpamos: cabeçalhos/rodapés repetidos, hifenização de fim de linha, números de página, e-mails, e **removemos referências bibliográficas e agradecimentos** (eles poluem a recuperação). Marcadores `<!--page:N-->` preservam a página de origem de cada trecho.

In [ ]:
REF_HEAD  = re.compile(r"^\s*(#{1,6}\s*)?[\*_]*\s*(\d+\.?\s*)?(references|referências|referencias|literature cited|bibliografia|bibliography|works cited)"
                       r"(\s+(bibliográficas|bibliograficas|citadas|cited))?\s*[\*_:]*\s*$", re.I)
SKIP_HEAD = re.compile(r"^\s*(#{1,6}\s*)?[\*_]*\s*(\d+\.?\s*)?(acknowledge?ments?|agradecimentos|conflicts? of interest|competing interests|author contributions|data availability|funding|publisher.?s note|ethics statement|generative ai statement|supplementary material|declarations)\s*[\*_:]*\s*$", re.I)
SECTION_LIKE = re.compile(r"^\s*(#{1,6}\s+\S.*|\*\*[^*]{2,80}\*\*\s*)$")
# Em dissertações cada capítulo tem suas referências: a remoção termina no próximo título de capítulo
CHAPTER_HEAD = re.compile(r"^\s*(#{1,6}\s*)?[\*_]*\s*(\d+\.?\s*)?(cap[íi]tulo|chapter|introdu|resumo|abstract|material|m[ée]todo|"
                          r"result|discuss|conclus|considera[çc][õo]es|anexo|ap[êe]ndice|appendix)", re.I)
PAGE_MARK = re.compile(r"^\s*<!--page:(\d+)-->\s*$")

def clean_heading(line):
    """'## **2** | **FACTORS ...**' -> '2 | FACTORS ...'. Devolve '' se não parecer um título de verdade."""
    h = re.sub(r"<[^>]+>|[#*_`]", "", line).strip(" |:-")
    h = re.sub(r"\s+", " ", h)
    return h[:120] if len(re.findall(r"[A-Za-zÀ-ú]", h)) >= 4 else ""

def slugify(s, n=40):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")[:n]

def strip_repeated_lines(pages):
    """Remove linhas curtas que se repetem em >50% das páginas (cabeçalhos/rodapés)."""
    if len(pages) < 4:
        return pages
    cnt = Counter()
    for p in pages:
        cnt.update({l.strip() for l in p.splitlines() if 0 < len(l.strip()) < 120})
    rep = {l for l, c in cnt.items() if c / len(pages) > 0.5}
    return ["\n".join(l for l in p.splitlines() if l.strip() not in rep) for p in pages]

# Fontes Adobe (ex.: revista Oryx) codificam algarismos "old-style" na área de uso privado do Unicode:
# U+F730..F739 = 0..9. Sem este mapa, "14 tapirs" vira " tapirs" e os números somem do RAG.
PUA_MAP = {0xF730 + i: str(i) for i in range(10)} | {0xF0B7: "•", 0xF020: " ", 0xF02D: "-"}

def clean_markdown(text):
    text = text.translate(PUA_MAP)
    text = re.sub(r"[-]", "", text)               # demais caracteres de uso privado
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)            # disper-\nsão -> dispersão
    text = re.sub(r"\S+@\S+\.\w+", "", text)                 # e-mails
    text = re.sub(r"<sup>.*?</sup>", "", text, flags=re.S)   # marcadores de afiliação/nota (¹ ²)
    text = re.sub(r"</?(sub|br|span|i|b|u|em|strong)\b[^>]*>", "", text)
    text = re.sub(r"\*\*==> picture.*?<==\*\*", "", text)     # placeholders de imagem do pymupdf4llm
    text = re.sub(r"(?m)^.*(\.\s?){5,}\s*\d+\s*$", "", text)   # linhas de sumário "Introdução ........ 12"
    out, mode = [], "keep"
    for line in text.splitlines():
        if PAGE_MARK.match(line):
            out.append(line.strip()); continue
        if REF_HEAD.match(line):
            mode = "refs"; continue                          # referências: até o próximo capítulo (ou o fim)
        if SKIP_HEAD.match(line):
            mode = "skip"; continue                          # agradecimentos etc.: até o próximo título
        if mode == "skip" and SECTION_LIKE.match(line):
            mode = "keep"
        if mode == "refs" and SECTION_LIKE.match(line) and CHAPTER_HEAD.match(line):
            mode = "keep"
        if mode != "keep":
            continue
        if re.fullmatch(r"\s*\d{1,4}\s*", line):             # número de página solto
            continue
        out.append(line)
    text = "\n".join(out)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

DOI_RE = re.compile(r"\b(10\.\d{4,9}/[^\s\"<>]+)", re.I)
def guess_metadata(text):
    head = text[:4000]
    doi = DOI_RE.search(head)
    years = [int(y) for y in re.findall(r"\b(19[89]\d|20[0-2]\d)\b", head)]
    title = ""
    for line in head.splitlines():
        t = re.sub(r"^[#\*\s_]+|[\*_\s]+$", "", line).strip()
        if len(t) > 25 and not t.lower().startswith(("doi", "http", "received", "accepted", "copyright", "©", "<!--")):
            title = t; break
    return {"doi": doi.group(1).rstrip(".,;)") if doi else "",
            "year": max(set(years), key=years.count) if years else None,
            "title": title[:250]}

def parse_pages(spec, n_pages):
    """'43-46' ou '1,3,5-7' (1-based, como no leitor de PDF) -> índices 0-based. Vazio = todas."""
    spec = str(spec or "").strip()
    if not spec:
        return list(range(n_pages))
    idx = []
    for part in spec.split(","):
        a, _, b = part.strip().partition("-")
        idx += range(int(a) - 1, int(b or a))
    return [i for i in idx if 0 <= i < n_pages]

def truthy(v):
    return str(v).strip().lower() in {"sim", "s", "true", "1", "yes", "y"}

def catalog_text(r):
    """Ficha de um trabalho da planilha (inclusive os sem PDF): permite responder 'que estudos existem sobre X?'."""
    campos = [("Tipo", "tipo_documento"), ("Autores", "authors"), ("Espécies", "especies"),
              ("Projeto", "projeto"), ("Local", "local"), ("DOI", "doi")]
    linhas = [f"Ficha de publicação do Refauna: {r['title']} ({r.get('year') or 's.d.'})."]
    linhas += [f"{rotulo}: {r[k]}." for rotulo, k in campos if str(r.get(k) or "").strip()]
    if not truthy(r.get("incluir_texto")):
        linhas.append("O texto completo deste trabalho não está na base; apenas estes metadados.")
    return " ".join(linhas)

In [ ]:
import pymupdf, pymupdf4llm, hashlib
import pandas as pd

# O modo "layout" do pymupdf4llm 2.x transforma texto em duas colunas em tabelas quebradas
# (testado com os PDFs do Refauna). O modo clássico extrai texto corrido muito melhor.
if hasattr(pymupdf4llm, "use_layout"):
    pymupdf4llm.use_layout(False)

# O manifesto decide o que entra: incluir_texto=sim, e 'paginas' recorta compilados (livro IUCN, boletim)
manifest = pd.read_csv(SOURCES_CSV, dtype=str, keep_default_na=False).to_dict("records")
print(f"sources.csv: {len(manifest)} trabalhos catalogados | "
      f"{sum(truthy(r['incluir_texto']) for r in manifest)} com texto para o RAG/treino")

def pdf_to_pages(path, page_idx):
    kw = dict(pages=page_idx, page_chunks=True)
    try:
        res = pymupdf4llm.to_markdown(str(path), use_ocr=False, **kw)   # sem OCR: todos os PDFs têm texto
    except TypeError:
        res = pymupdf4llm.to_markdown(str(path), **kw)                  # versões antigas não têm use_ocr
    out = []
    for i, p in zip(page_idx, res):
        m = p.get("metadata", {})
        out.append((m.get("page_number") or m.get("page") or i + 1, p["text"]))
    return out

docs, seen = [], {}
for r in manifest:
    if not truthy(r["incluir_texto"]):
        continue
    pdf = PDF_DIR / r["filename"]
    if not pdf.exists():
        print(f"❌ {r['doc_id']}: arquivo não encontrado ({r['filename']})"); continue
    n_pages = pymupdf.open(pdf).page_count
    page_idx = parse_pages(r["paginas"], n_pages)
    key = (hashlib.md5(pdf.read_bytes()).hexdigest(), tuple(page_idx))   # trava contra PDFs duplicados
    if key in seen:
        print(f"⚠️ {r['doc_id']}: mesmo conteúdo de {seen[key]} — ignorado"); continue
    seen[key] = r["doc_id"]

    pages = pdf_to_pages(pdf, page_idx)
    page_texts = strip_repeated_lines([t for _, t in pages])
    raw = "\n".join(f"<!--page:{n}-->\n{t}" for (n, _), t in zip(pages, page_texts))
    text = clean_markdown(raw)
    body = re.sub(r"<!--page:\d+-->", "", text).strip()
    if len(body) < 800:
        print(f"⚠️ {r['doc_id']}: pouco texto ({len(body)} chars). PDF escaneado? — ignorado."); continue
    meta = guess_metadata(body)
    doc = {
        "doc_id": r["doc_id"], "filename": r["filename"],
        "title": r["title"].strip() or meta["title"] or pdf.stem,
        "authors": r["authors"].strip(),
        "year": int(r["year"]) if r["year"].strip().isdigit() else meta["year"],
        "doi": r["doi"].strip() or meta["doi"],
        "tipo_documento": r.get("tipo_documento", ""),
        "license": r.get("license", "").strip() or "a verificar",
        "redistributable": truthy(r.get("redistributable", "")),
        "paginas": r["paginas"], "n_pages": len(pages), "n_chars": len(body),
    }
    (MD_DIR / f"{doc['doc_id']}.md").write_text(text, encoding="utf-8")
    docs.append(doc)

write_jsonl(DATA_DIR / "docs.jsonl", docs)
df_docs = pd.DataFrame(docs)
print(f"✅ {len(docs)} documentos | {df_docs.n_pages.sum()} páginas | {df_docs.n_chars.sum()/1e3:.0f} mil caracteres")
df_docs[["doc_id", "title", "year", "tipo_documento", "paginas", "n_pages", "n_chars"]]

> 🔎 **Confira a tabela.** Metadados errados? Corrija no `sources.csv` (ele é a fonte da verdade) e rode a célula de novo. Abra 1–2 arquivos em `data/markdown/` para ver se a limpeza ficou boa. Conseguiu mais PDFs? Coloque em `training-files/`, preencha `filename` e marque `incluir_texto = sim`.

## 2. Chunking por seção
Agrupamos parágrafos da mesma seção até ~400 tokens (medidos com o tokenizer do próprio Gemma 4), com ~60 tokens de sobreposição em frases. Cada chunk guarda documento, seção e páginas — é isso que permite **citar a fonte** na resposta.

In [ ]:
CANON = [("abstract", r"abstract|resumo|summary"), ("introduction", r"introdu"),
         ("methods", r"method|métod|metod|material|study area|área de estudo|area de estudo"),
         ("results", r"result"), ("discussion", r"discuss"), ("conclusion", r"conclus|final remarks|considera")]
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-ZÀ-Ú\(\[])")

def canon_section(h):
    h = h.lower()
    for name, pat in CANON:
        if re.search(pat, h):
            return name
    return "other"

REF_SIGNS = re.compile(r"(?<![\(\[])\b(19|20)\d{2}[a-z]?\.\s|\b\d+\s?\(\d+\)\s?:\s?\d+|doi\.org/|\bIn press\b", re.I)
def looks_like_references(text, threshold=5):
    """Lista de referências sem título (comum em boletins e na Frontiers): muitas entradas 'Autor. 2012. Título. Revista 7(4): 796'."""
    return len(REF_SIGNS.findall(text)) >= threshold

def iter_paragraphs(md_text):
    """Gera (seção, página, parágrafo) a partir do Markdown limpo."""
    section, page, buf, buf_page = "Início", 1, [], 1
    for line in md_text.splitlines():
        m = PAGE_MARK.match(line)
        if m:
            page = int(m.group(1)); continue
        if SECTION_LIKE.match(line):
            h = clean_heading(line)
            if not h:                       # "## 39", "**164**": número de página formatado como título
                continue
            if buf:
                yield section, buf_page, " ".join(buf); buf = []
            section = h
            continue
        if not line.strip():
            if buf:
                yield section, buf_page, " ".join(buf); buf = []
            continue
        if not buf:
            buf_page = page
        buf.append(line.strip())
    if buf:
        yield section, buf_page, " ".join(buf)

def chunk_document(doc, md_text, count, max_tok=CHUNK_TOKENS, overlap=CHUNK_OVERLAP, min_tok=MIN_CHUNK_TOKENS):
    units = []                                   # (seção, página, texto, n_tokens)
    for sec, pg, para in iter_paragraphs(md_text):
        t = count(para)
        if t > max_tok:
            units += [(sec, pg, s, count(s)) for s in SENT_SPLIT.split(para) if s.strip()]
        else:
            units.append((sec, pg, para, t))

    chunks, cur = [], []
    def emit():
        text = "\n".join(u[2] for u in cur).strip()
        nt = count(text)
        sec = cur[-1][0]
        if looks_like_references(text):
            return
        if nt >= min_tok or (canon_section(sec) == "abstract" and nt > 20):
            chunks.append({
                "chunk_id": f"{doc['doc_id']}_c{len(chunks):04d}", "idx": len(chunks),
                "doc_id": doc["doc_id"], "title": doc["title"], "authors": doc.get("authors", ""),
                "year": doc.get("year"), "doi": doc.get("doi", ""),
                "license": doc.get("license", ""), "redistributable": doc.get("redistributable", False),
                "section": sec, "section_canon": canon_section(sec),
                "page_start": min(u[1] for u in cur), "page_end": max(u[1] for u in cur),
                "text": text, "n_tokens": nt,
            })
    def tail_of(text):
        tail, t = [], 0
        for s in reversed(SENT_SPLIT.split(text)):
            c = count(s)
            if t + c > overlap:
                break
            tail.insert(0, s); t += c
        return " ".join(tail), t

    for u in units:
        if cur and (u[0] != cur[-1][0] or sum(x[3] for x in cur) + u[3] > max_tok):
            same_section = u[0] == cur[-1][0]
            emit()
            tail_text, tail_tok = tail_of(cur[-1][2]) if same_section else ("", 0)
            cur = [(cur[-1][0], cur[-1][1], tail_text, tail_tok)] if tail_text else []
        cur.append(u)
    if cur:
        emit()
    return chunks

In [ ]:
from transformers import AutoTokenizer
try:
    _tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    n_tokens = lambda s: len(_tok.encode(s, add_special_tokens=False))
except Exception as e:
    print("Tokenizer indisponível, usando aproximação por palavras:", e)
    n_tokens = lambda s: int(len(s.split()) * 1.4)

chunks = []
for doc in docs:
    md_text = (MD_DIR / f"{doc['doc_id']}.md").read_text(encoding="utf-8")
    chunks += chunk_document(doc, md_text, count=n_tokens)

# Fichas do catálogo: 1 trecho curto por trabalho da planilha (inclusive os ~70 sem PDF).
# Entram no índice do RAG, mas NÃO são usadas para gerar perguntas de treino.
for r in manifest:
    t = catalog_text(r)
    chunks.append({
        "chunk_id": f"{r['doc_id']}_ficha", "idx": -100, "doc_id": r["doc_id"], "title": r["title"],
        "authors": r["authors"], "year": int(r["year"]) if r["year"].strip().isdigit() else None,
        "doi": r["doi"], "license": "metadados", "redistributable": True,
        "section": "Ficha do catálogo", "section_canon": "catalog", "page_start": 0, "page_end": 0,
        "text": t, "n_tokens": n_tokens(t),
    })

write_jsonl(DATA_DIR / "chunks.jsonl", chunks)
chunk_by_id = {c["chunk_id"]: c for c in chunks}
df_chunks = pd.DataFrame(chunks)
print(f"✅ {len(chunks)} chunks | tokens/chunk: média {df_chunks.n_tokens.mean():.0f}, "
      f"p10 {df_chunks.n_tokens.quantile(.1):.0f}, p90 {df_chunks.n_tokens.quantile(.9):.0f}")
print(df_chunks.section_canon.value_counts().to_string())

## 3. RAG — índice híbrido (denso + léxico)

```
pergunta ─┬─► EmbeddingGemma (encode_query) ─► ChromaDB (top-20, cosseno) ─┐
          └─► BM25 (top-20) ───────────────────────────────────────────────┴─► RRF ─► top-4 ─► prompt numerado [1]..[4]
```

- **Denso (EmbeddingGemma 300M):** entende paráfrases e perguntas em PT-BR sobre artigos em inglês (multilíngue).
- **Léxico (BM25):** acerta **nomes científicos** e siglas (*Alouatta guariba*, *Dasyprocta leporina*, PNT), onde a busca vetorial costuma falhar.
- **Reciprocal Rank Fusion:** combina os dois rankings sem precisar calibrar escalas.

O mesmo índice será usado no Mac. **Regra de ouro:** o modelo de embedding na consulta tem que ser o mesmo da indexação.

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from rank_bm25 import BM25Okapi

# EmbeddingGemma não suporta float16 -> float32
embedder = SentenceTransformer(EMBED_MODEL, device="cuda", model_kwargs={"torch_dtype": torch.float32})

def index_text(c):
    return f"{c['title']} — {c['section']}\n{c['text']}"

t0 = time.time()
doc_emb = embedder.encode_document([index_text(c) for c in chunks], batch_size=32,
                                   normalize_embeddings=True, show_progress_bar=True)
np.save(DATA_DIR / "chunk_embeddings.npy", doc_emb)
print(f"Embeddings: {doc_emb.shape} em {time.time()-t0:.0f}s")

# Chroma em disco local (SQLite no Drive é lento/instável) e depois copiamos para o Drive
CHROMA_LOCAL = LOCAL_DIR / "chroma"
shutil.rmtree(CHROMA_LOCAL, ignore_errors=True)
chroma = chromadb.PersistentClient(path=str(CHROMA_LOCAL))
col = chroma.create_collection("refauna", metadata={"hnsw:space": "cosine"})

def chroma_meta(c):
    m = {k: (c.get(k) or "") for k in ("doc_id", "title", "authors", "doi", "section", "section_canon", "license")}
    m.update(year=int(c.get("year") or 0), page_start=int(c["page_start"]), page_end=int(c["page_end"]))
    return m

for i in range(0, len(chunks), 500):
    b = chunks[i:i + 500]
    col.add(ids=[c["chunk_id"] for c in b], embeddings=doc_emb[i:i + 500].tolist(),
            documents=[c["text"] for c in b], metadatas=[chroma_meta(c) for c in b])
shutil.copytree(CHROMA_LOCAL, DATA_DIR / "chroma", dirs_exist_ok=True)
print("Chroma:", col.count(), "vetores | chromadb", chromadb.__version__, "(use a MESMA versão no Mac)")

In [ ]:
STOP = set(("a o e de da do das dos em no na nos nas um uma para por com que se ao os as sao "
            "the of and to in is for on with by as are was were be this that from at an or").split())

def bm25_tokens(s):
    s = unicodedata.normalize("NFKD", s.lower()).encode("ascii", "ignore").decode()
    return [t for t in re.findall(r"[a-z0-9]+", s) if t not in STOP and len(t) > 1]

def rrf_fuse(rankings, rrf_k=60):
    fused = defaultdict(float)
    for ranking in rankings:
        for rank, cid in enumerate(ranking):
            fused[cid] += 1.0 / (rrf_k + rank + 1)
    return sorted(fused, key=fused.get, reverse=True)

def select_diverse(ranked_ids, by_id, k, exclude_ids=()):
    """Remove excluídos e chunks vizinhos (sobrepostos) do mesmo documento."""
    out = []
    for cid in ranked_ids:
        if cid in exclude_ids:
            continue
        c = by_id[cid]
        if any(o["doc_id"] == c["doc_id"] and abs(o["idx"] - c["idx"]) <= 1 for o in out):
            continue
        out.append(c)
        if len(out) == k:
            break
    return out

In [ ]:
bm25 = BM25Okapi([bm25_tokens(index_text(c)) for c in chunks])
chunk_ids = [c["chunk_id"] for c in chunks]

def hybrid_search(query, k=TOP_K, n_candidates=20, exclude_ids=()):
    q = embedder.encode_query(query, normalize_embeddings=True)
    dense = col.query(query_embeddings=[q.tolist()], n_results=min(n_candidates, len(chunks)))["ids"][0]
    scores = bm25.get_scores(bm25_tokens(query))
    lexical = [chunk_ids[i] for i in np.argsort(scores)[::-1][:n_candidates]]
    return select_diverse(rrf_fuse([dense, lexical]), chunk_by_id, k, exclude_ids)

for q in ["O que é refaunação e por que ela importa para a Mata Atlântica?",
          "Quais animais foram reintroduzidos no Parque Nacional da Tijuca?",
          "Qual o papel das cutias na dispersão de sementes?"]:
    print(f"\n❓ {q}")
    for i, c in enumerate(hybrid_search(q), 1):
        print(f"  [{i}] {short_ref(c)} · {c['section']} · p.{c['page_start']} :: {c['text'][:110]}...")

## 4. Dataset sintético no formato RAG (estilo RAFT)

Um LLM maior (Claude) lê cada chunk e escreve perguntas/respostas em PT-BR. Depois **montamos** os exemplos de treino:

| Tipo | % | Contexto | Resposta ensinada |
|---|---|---|---|
| `rag` | ~60% | trecho correto (oráculo) + 3 distratores recuperados, embaralhados | resposta fiel citando `[n]` do oráculo |
| `unanswerable` | ~15% | 4 trechos que **não** contêm a resposta | abstenção ("não encontrei nos documentos do Refauna") |
| `closed_book` | ~25% | nenhum | resposta conceitual, sem números específicos |

Os distratores vêm da **própria busca híbrida**: o modelo treina com o mesmo tipo de ruído que verá na inferência.

In [ ]:
def parse_json(txt):
    if not txt:
        return None
    txt = re.sub(r"^```(?:json)?|```$", "", txt.strip(), flags=re.M).strip()
    a, b = txt.find("{"), txt.rfind("}")
    if a < 0 or b <= a:
        return None
    try:
        return json.loads(txt[a:b + 1])
    except json.JSONDecodeError:
        return None

GEN_SYSTEM = ("Você cria dados de treinamento de alta qualidade, em português do Brasil, para um assistente "
              "especializado em refaunação e conservação da Mata Atlântica, a partir de trechos de artigos "
              "científicos do projeto Refauna. Você é fiel ao texto e nunca inventa informações.")

def gen_prompt(c, n=QUESTIONS_PER_CHUNK):
    return f"""Documento: {c['title']} ({c.get('year') or 's.d.'}) — seção: {c['section']}

TRECHO:
\"\"\"
{c['text']}
\"\"\"

Tarefa:
1. Gere {n} pares pergunta-resposta que possam ser respondidos SOMENTE com o TRECHO.
   - Varie o tipo: factual, conceitual, comparativa, metodologica, didatica (explicação para leigos).
   - Perguntas autocontidas, como um usuário real faria: nomeie espécie, local ou estudo. Nunca escreva "segundo o trecho", "neste estudo" ou "o artigo".
   - Respostas com 2 a 6 frases, em PT-BR (mesmo que o trecho esteja em inglês), didáticas e fiéis.
   - Coloque a marca [FONTE] ao final de cada frase da resposta que usa informação do trecho.
   - Nomes científicos em itálico Markdown (*Genus species*). Não invente números, datas ou locais.
2. Gere 1 pergunta conceitual geral ("closed_book") inspirada no tema, que um especialista responderia sem consultar o artigo; resposta conceitual sem números específicos e sem [FONTE]. Use null se não fizer sentido.
3. Se o trecho for pouco informativo (tabela quebrada, legenda solta, lista de referências), devolva {{"qa": [], "closed_book": null}}.

Responda APENAS com JSON válido neste formato:
{{"qa": [{{"question": "...", "answer": "...", "question_type": "factual|conceitual|comparativa|metodologica|didatica"}}],
  "closed_book": {{"question": "...", "answer": "..."}}}}"""

In [ ]:
import anthropic
import concurrent.futures as cf

llm = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def call_llm(system, user, max_tokens=1500, retries=5):
    for attempt in range(retries):
        try:
            r = llm.messages.create(model=GEN_MODEL, max_tokens=max_tokens, system=system,
                                    messages=[{"role": "user", "content": user}])
            return "".join(b.text for b in r.content if b.type == "text")
        except Exception as e:
            wait = 2 ** attempt + random.random()
            print(f"  tentativa {attempt+1} falhou ({type(e).__name__}); aguardando {wait:.0f}s")
            time.sleep(wait)
    return None

# Seleção dos chunks: informativos (>=150 tokens) e com cobertura de TODOS os documentos (round-robin)
by_doc = defaultdict(list)
for c in sorted(chunks, key=lambda c: c["chunk_id"]):
    if c["n_tokens"] >= 150 and c["section_canon"] != "catalog":
        by_doc[c["doc_id"]].append(c)
for lst in by_doc.values():
    random.Random(SEED).shuffle(lst)
limit = MAX_CHUNKS_FOR_GEN or sum(map(len, by_doc.values()))
selected = []
while len(selected) < limit and any(by_doc.values()):
    for d in list(by_doc):
        if by_doc[d] and len(selected) < limit:
            selected.append(by_doc[d].pop())

est_in = sum(c["n_tokens"] for c in selected) + 700 * len(selected)
est_out = 600 * len(selected)
print(f"{len(selected)} chamadas ao {GEN_MODEL} | ~{est_in/1e6:.2f}M tokens de entrada, ~{est_out/1e6:.2f}M de saída.")
print("Confira o preço por token do modelo antes de seguir. Para um teste barato, use MAX_CHUNKS_FOR_GEN = 30.")

In [ ]:
CACHE = DATA_DIR / "gen_cache.jsonl"          # retomável: chunks já processados não são regerados
done = {r["chunk_id"]: r for r in read_jsonl(CACHE)} if CACHE.exists() else {}
todo = [c for c in selected if c["chunk_id"] not in done]
print(f"Em cache: {len(done)} | a gerar: {len(todo)}")

def work(c):
    return {"chunk_id": c["chunk_id"], "result": parse_json(call_llm(GEN_SYSTEM, gen_prompt(c)))}

with open(CACHE, "a", encoding="utf-8") as f, cf.ThreadPoolExecutor(max_workers=4) as ex:
    futures = [ex.submit(work, c) for c in todo]
    for i, fut in enumerate(cf.as_completed(futures), 1):
        r = fut.result()
        f.write(json.dumps(r, ensure_ascii=False) + "\n"); f.flush()
        done[r["chunk_id"]] = r
        if i % 25 == 0 or i == len(todo):
            print(f"  {i}/{len(todo)}")

ok = sum(1 for r in done.values() if r["result"] is not None)
print(f"✅ {ok}/{len(done)} respostas em JSON válido")

In [ ]:
VALID_QT = {"factual", "conceitual", "comparativa", "metodologica", "didatica"}

def collect_items(done, chunk_by_id, per_chunk=QUESTIONS_PER_CHUNK):
    qa_items, cb_items = [], []
    for cid, r in sorted(done.items()):
        if cid not in chunk_by_id:
            continue
        res = r.get("result") or {}
        for qa in (res.get("qa") or [])[:per_chunk]:
            q, a = (qa.get("question") or "").strip(), (qa.get("answer") or "").strip()
            if len(q) > 10 and len(a) > 20:
                qt = qa.get("question_type") if qa.get("question_type") in VALID_QT else "factual"
                qa_items.append({"chunk_id": cid, "question": q, "answer": a, "question_type": qt})
        cb = res.get("closed_book") or {}
        if cb.get("question") and cb.get("answer"):
            cb_items.append({"chunk_id": cid, "question": cb["question"].strip(),
                             "answer": cb["answer"].replace("[FONTE]", "").strip(), "question_type": "conceitual"})
    return qa_items, cb_items

def insert_citations(answer, n):
    ans = answer.replace("[FONTE]", f"[{n}]")
    ans = re.sub(r"\s+\[(\d+)\]", r" [\1]", ans)
    ans = re.sub(r"(\[\d+\])(\s*\1)+", r"\1", ans)           # [2] [2] -> [2]
    if f"[{n}]" not in ans:
        ans = ans.rstrip() + f" [{n}]"
    return ans

def make_example(kind, item, src, contexts, oracle_index, answer):
    msgs = build_messages(item["question"], contexts) + [{"role": "assistant", "content": answer}]
    return {"id": None, "doc_id": src["doc_id"], "kind": kind, "question_type": item["question_type"],
            "question": item["question"], "reference": answer, "messages": msgs,
            "oracle_index": oracle_index, "source_chunk_ids": [item["chunk_id"]],
            "context_chunk_ids": [c["chunk_id"] for c in contexts] if contexts else []}

In [ ]:
def pick_distractors(question, oracle, n):
    """Distratores 'difíceis' vindos da busca híbrida, evitando o oráculo e seus vizinhos."""
    out = []
    for c in hybrid_search(question, k=n + 8, exclude_ids={oracle["chunk_id"]}):
        if c["doc_id"] == oracle["doc_id"] and abs(c["idx"] - oracle["idx"]) <= 2:
            continue
        out.append(c)
        if len(out) == n:
            break
    pool = ([c for c in chunks if c["doc_id"] != oracle["doc_id"] and c["section_canon"] != "catalog"]
            or [c for c in chunks if c is not oracle])
    while len(out) < n and pool:
        c = random.choice(pool)
        if c not in out:
            out.append(c)
    return out

def pick_unrelated(question, oracle, n):
    """Para 'unanswerable': trechos de OUTROS documentos fora do top-10 da busca
    (os artigos do Refauna se sobrepõem muito; o top-10 poderia conter a resposta)."""
    top = {c["chunk_id"] for c in hybrid_search(question, k=10)}
    pool = [c for c in chunks if c["doc_id"] != oracle["doc_id"] and c["chunk_id"] not in top
            and c["n_tokens"] >= 100 and c["section_canon"] != "catalog"]
    return random.sample(pool, min(n, len(pool)))

qa_items, cb_items = collect_items(done, chunk_by_id)
random.Random(SEED).shuffle(qa_items)
n_un = int(0.20 * len(qa_items))                  # 20% do pool QA -> ~15% do total
examples = []
for i, it in enumerate(qa_items):
    oracle = chunk_by_id[it["chunk_id"]]
    if i < n_un:
        ctx = pick_unrelated(it["question"], oracle, TOP_K)
        if len(ctx) == TOP_K:
            examples.append(make_example("unanswerable", it, oracle, ctx, None, random.choice(ABSTAIN_ANSWERS)))
            continue
    ctx = pick_distractors(it["question"], oracle, TOP_K - 1) + [oracle]
    random.shuffle(ctx)
    n = ctx.index(oracle) + 1
    examples.append(make_example("rag", it, oracle, ctx, n, insert_citations(it["answer"], n)))

n_cb = min(len(cb_items), round(len(qa_items) / 3))  # ~25% do total
for it in random.Random(SEED).sample(cb_items, n_cb):
    examples.append(make_example("closed_book", it, chunk_by_id[it["chunk_id"]], None, None, it["answer"]))

for i, e in enumerate(examples):
    e["id"] = f"sft_{i:05d}"
print(f"✅ {len(examples)} exemplos:", dict(Counter(e["kind"] for e in examples)))
print("\n--- Exemplo RAG (mensagem do usuário + resposta) ---")
ex = next(e for e in examples if e["kind"] == "rag")
print(ex["messages"][1]["content"][:1200], "\n\n>>>", ex["messages"][2]["content"])

## 5. Curadoria, revisão humana e split por documento
1. Filtros automáticos (idioma, citação ausente, respostas vazias).
2. Deduplicação de perguntas por similaridade semântica (> 0,92).
3. **Revisão humana**: exportamos 150 exemplos para CSV. Baixe, marque `ok` = `s`/`n` e, se quiser, escreva uma `correcao`. Suba como `refauna-slm/revisao_feita.csv` e rode a célula seguinte. A % aprovada é um ótimo número para a palestra.
4. **Split por documento**: ~15% dos artigos ficam inteiramente no teste → mede se o modelo generaliza para fatos que nunca viu no treino (o RAG precisa trazê-los).

In [ ]:
def passes_filters(e):
    ans = e["messages"][-1]["content"]
    if len(ans) < 20 or not is_ptbr(ans):
        return False
    if e["kind"] == "rag" and not re.search(r"\[\d\]", ans):
        return False
    return True

before = len(examples)
examples = [e for e in examples if passes_filters(e)]

q_emb = embedder.encode_query([e["question"] for e in examples], normalize_embeddings=True,
                              convert_to_tensor=True, batch_size=64)
sim = q_emb @ q_emb.T
removed = torch.zeros(len(examples), dtype=torch.bool, device=sim.device)
keep = []
for i in range(len(examples)):
    if removed[i]:
        continue
    keep.append(i)
    dup = sim[i] > 0.92
    dup[: i + 1] = False
    removed |= dup
examples = [examples[i] for i in keep]
print(f"Filtros + dedup: {before} -> {len(examples)} exemplos")

review = pd.DataFrame([{"id": e["id"], "kind": e["kind"], "pergunta": e["question"],
                        "contexto": e["messages"][1]["content"], "resposta": e["messages"][-1]["content"],
                        "ok": "", "correcao": ""} for e in random.Random(SEED).sample(examples, min(150, len(examples)))])
review.to_csv(PROJECT_DIR / "revisao_amostra.csv", index=False)
print("📝 Amostra para revisão salva em refauna-slm/revisao_amostra.csv")

In [ ]:
REVIEW_DONE = PROJECT_DIR / "revisao_feita.csv"
if REVIEW_DONE.exists():
    rv = pd.read_csv(REVIEW_DONE).fillna("")
    rv["ok"] = rv["ok"].astype(str).str.strip().str.lower()
    marked = rv[rv["ok"].isin(["s", "n"])]
    print(f"Aprovação humana: {(marked.ok == 's').mean():.0%} de {len(marked)} revisados")
    fix = {r.id: r.correcao.strip() for r in marked.itertuples() if r.ok == "n" and r.correcao.strip()}
    drop = {r.id for r in marked.itertuples() if r.ok == "n" and not r.correcao.strip()}
    for e in examples:
        if e["id"] in fix:
            e["messages"][-1]["content"] = e["reference"] = fix[e["id"]]
    examples = [e for e in examples if e["id"] not in drop]
    print(f"Corrigidos: {len(fix)} | removidos: {len(drop)} | total: {len(examples)}")
else:
    print("Sem revisao_feita.csv — seguindo sem revisão humana (recomendo fazer antes do treino final).")

In [ ]:
def split_by_doc(examples, test_frac=0.15, val_frac=0.10, seed=SEED):
    doc_ids = sorted({e["doc_id"] for e in examples})
    random.Random(seed).shuffle(doc_ids)
    test_docs = set(doc_ids[: max(1, round(test_frac * len(doc_ids)))])
    test = [e for e in examples if e["doc_id"] in test_docs]
    rest = [e for e in examples if e["doc_id"] not in test_docs]
    random.Random(seed).shuffle(rest)
    n_val = max(20, int(val_frac * len(rest)))
    return rest[n_val:], rest[:n_val], test, test_docs

In [ ]:
train, val, test, test_docs = split_by_doc(examples)
assert not ({e["doc_id"] for e in train} & test_docs), "vazamento de documento entre treino e teste!"
for name, rows in [("train", train), ("val", val), ("test", test)]:
    write_jsonl(DATA_DIR / "sft" / f"{name}.jsonl", rows)
    print(f"{name:5s}: {len(rows):5d}", dict(Counter(e["kind"] for e in rows)))
print("Documentos de teste:", sorted(test_docs))

### 📏 Avaliação da recuperação (antes de qualquer treino)
Para cada pergunta do teste, o trecho de origem aparece entre os 4 primeiros recuperados? (aceitamos o chunk vizinho, por causa da sobreposição)

In [ ]:
def retrieval_metrics(items, k=TOP_K):
    hit = doc_hit = 0; rr = 0.0
    for e in items:
        src = chunk_by_id[e["source_chunk_ids"][0]]
        res = hybrid_search(e["question"], k=20)
        pos = next((i for i, c in enumerate(res) if c["doc_id"] == src["doc_id"] and abs(c["idx"] - src["idx"]) <= 1), None)
        hit += pos is not None and pos < k
        rr += 1 / (pos + 1) if pos is not None else 0
        doc_hit += any(c["doc_id"] == src["doc_id"] for c in res[:k])
    n = len(items)
    return {"n": n, f"recall@{k}": hit / n, "mrr@20": rr / n, f"doc_recall@{k}": doc_hit / n}

ret_items = [e for e in test if e["kind"] in ("rag", "unanswerable")]
retrieval_report = retrieval_metrics(ret_items)
print(retrieval_report)

## 6. Fine-tuning do Gemma 4 E2B com QLoRA (Unsloth)

- **QLoRA:** modelo-base carregado em 4-bit (congelado) + adaptadores LoRA treináveis em 16-bit. Mude `LOAD_IN_4BIT = False` na configuração para comparar com LoRA 16-bit.
- Só as camadas de **linguagem** (atenção + MLP) recebem LoRA; visão e áudio ficam de fora.
- Template **`gemma-4`** (sem *thinking*), recomendado para os modelos pequenos.
- `train_on_responses_only`: a loss só considera a resposta do modelo, não o system prompt nem o contexto.

> ℹ️ Loss inicial na faixa de **13–15** é normal no Gemma 4 E2B/E4B (peculiaridade de modelos multimodais, documentada pela Unsloth). Observe a **queda** e a `eval_loss`.

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL, dtype=None, max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=LOAD_IN_4BIT, full_finetuning=False, token=HF_TOKEN,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none", random_state=SEED,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
tok = getattr(tokenizer, "tokenizer", tokenizer)   # tokenizer de texto (se vier um processor multimodal)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis: {trainable/1e6:.1f}M de {total/1e9:.2f}B ({100*trainable/total:.2f}%)")

In [ ]:
from datasets import Dataset

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
                     for m in batch["messages"]]}

def build_ds(rows):
    ds = Dataset.from_list([{"messages": e["messages"]} for e in rows]).map(to_text, batched=True, remove_columns=["messages"])
    n0 = len(ds)
    ds = ds.filter(lambda x: len(tok(x["text"], add_special_tokens=False)["input_ids"]) <= MAX_SEQ_LEN)
    if len(ds) < n0:
        print(f"  {n0-len(ds)} exemplos acima de {MAX_SEQ_LEN} tokens removidos")
    return ds

train_ds, val_ds = build_ds(train), build_ds(val)
print(f"train={len(train_ds)} val={len(val_ds)}\n")
print("Como o Gemma 4 enxerga um exemplo de treino:\n")
print(train_ds[0]["text"][:1500])

In [ ]:
from trl import SFTTrainer, SFTConfig

CKPT_DIR = LOCAL_DIR / RUN_NAME / "checkpoints"
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text", max_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS, learning_rate=LR, warmup_ratio=0.05,
        lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.001,
        logging_steps=5, eval_strategy="steps", eval_steps=50,
        save_strategy="steps", save_steps=50, save_total_limit=3,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        seed=SEED, output_dir=str(CKPT_DIR), report_to="none",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part="<|turn>user\n", response_part="<|turn>model\n")

# Verificação do mascaramento: deve sobrar só a resposta do modelo
labels = trainer.train_dataset[0]["labels"]
print(tok.decode([t for t in labels if t != -100])[:600])

In [ ]:
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer_stats = trainer.train()
train_minutes = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_reserved() / 1024**3

train_report = {
    "run": RUN_NAME, "load_in_4bit": LOAD_IN_4BIT, "lora_r": LORA_R, "epochs": EPOCHS, "lr": LR,
    "train_examples": len(train_ds), "minutes": round(train_minutes, 1), "peak_vram_gb": round(peak_gb, 2),
    "trainable_params_m": round(trainable / 1e6, 1), "gpu": torch.cuda.get_device_name(0),
    "best_eval_loss": trainer.state.best_metric,
}
(OUT_DIR / RUN_NAME).mkdir(parents=True, exist_ok=True)
json.dump(train_report, open(OUT_DIR / RUN_NAME / "train_report.json", "w"), indent=2)
json.dump(trainer.state.log_history, open(OUT_DIR / RUN_NAME / "log_history.json", "w"))
print(json.dumps(train_report, indent=2))

In [ ]:
import matplotlib.pyplot as plt
hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(8, 4))
plt.plot(*zip(*tr), label="train loss", alpha=.7)
if ev:
    plt.plot(*zip(*ev), "o-", label="eval loss")
plt.xlabel("passo"); plt.ylabel("loss"); plt.title(f"Gemma 4 E2B · {RUN_NAME}"); plt.legend(); plt.grid(alpha=.3)
plt.savefig(OUT_DIR / RUN_NAME / "loss.png", dpi=160, bbox_inches="tight"); plt.show()

ADAPTER_DIR = OUT_DIR / RUN_NAME / "lora"
model.save_pretrained(str(ADAPTER_DIR)); tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Adapter salvo em", ADAPTER_DIR)

### RAG ponta a ponta no Colab
`ask()` faz o pipeline completo: busca híbrida → prompt numerado → geração. Com `use_adapter=False` usamos o **mesmo modelo sem o LoRA** (`disable_adapter`) — assim comparamos base vs fine-tunado sem carregar dois modelos na GPU.

In [ ]:
FastModel.for_inference(model)

def _apply(messages):
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                         return_dict=True, return_tensors="pt")

def generate(messages, max_new_tokens=512, temperature=0.3):
    try:
        inputs = _apply(messages)
    except Exception:   # alguns processors exigem conteúdo no formato multimodal
        inputs = _apply([{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages])
    inputs = inputs.to("cuda")
    kw = dict(max_new_tokens=max_new_tokens, use_cache=True)   # use_cache=True é obrigatório no Gemma 4 E2B
    kw.update(do_sample=True, temperature=temperature, top_p=0.95, top_k=64) if temperature > 0 else kw.update(do_sample=False)
    with torch.inference_mode():
        out = model.generate(**inputs, **kw)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def ask(question, use_rag=True, use_adapter=True, k=TOP_K, temperature=0.3, show=True):
    ctx = hybrid_search(question, k=k) if use_rag else None
    msgs = build_messages(question, ctx)
    t0 = time.time()
    if use_adapter:
        answer = generate(msgs, temperature=temperature)
    else:
        with model.disable_adapter():
            answer = generate(msgs, temperature=temperature)
    dt = time.time() - t0
    if show:
        print(f"⏱ {dt:.1f}s\n{answer}")
        for i, c in enumerate(ctx or [], 1):
            print(f"   [{i}] {short_ref(c)} · {c['section']} · p.{c['page_start']}")
    return {"answer": answer, "contexts": ctx or [], "latency_s": dt}

q = "Por que a reintrodução de dispersores de sementes é importante para a regeneração da Mata Atlântica?"
for label, rag, ad in [("A · base", False, False), ("B · base + RAG", True, False),
                       ("C · fine-tunado", False, True), ("D · fine-tunado + RAG", True, True)]:
    print(f"\n{'='*20} {label} {'='*20}")
    ask(q, use_rag=rag, use_adapter=ad)

## 7. Avaliação — 4 configurações

| | sem RAG | com RAG |
|---|---|---|
| **base** | A | B |
| **fine-tunado** | C | D |

- **Trilha 1 (ponta a ponta):** perguntas dos documentos de teste + gold set, com recuperação real. Um juiz LLM dá notas de **correção** (vs. referência) e **fidelidade** ao contexto (alucinação). Medimos também **citação correta**, **% PT-BR** e latência.
- **Trilha 2 (abstenção):** com contextos sem a resposta, o modelo admite que não sabe?

Geração gulosa (`temperature=0`) para ser reprodutível.

In [ ]:
JUDGE_SYSTEM = "Você é um avaliador rigoroso de respostas sobre ecologia e conservação. Responda apenas com JSON."

def judge(question, reference, answer, contexts=None):
    ctx_block = f"\nCONTEXTO DISPONÍVEL AO MODELO:\n{format_context(contexts)}\n" if contexts else ""
    prompt = f"""PERGUNTA: {question}

RESPOSTA DE REFERÊNCIA: {reference}
{ctx_block}
RESPOSTA DO MODELO: {answer}

Avalie:
- "correcao" (1-5): a resposta do modelo transmite o mesmo conteúdo factual da referência? 5 = equivalente e completa; 3 = parcialmente correta; 1 = errada, irrelevante ou em outro idioma.
- "fidelidade" (1-5 ou null): {"tudo o que o modelo afirma está apoiado no CONTEXTO? 5 = nada inventado; 1 = muitas informações sem apoio." if contexts else "use null (não havia contexto)."}
Responda só com JSON: {{"correcao": int, "fidelidade": int|null, "comentario": "uma frase"}}"""
    return parse_json(call_llm(JUDGE_SYSTEM, prompt, max_tokens=300))

N_EVAL = 60
rng = random.Random(SEED)
test_rag = [e for e in test if e["kind"] == "rag"]
test_cb = [e for e in test if e["kind"] == "closed_book"]
test_un = [e for e in test if e["kind"] == "unanswerable"]
eval_items = rng.sample(test_rag, min(len(test_rag), int(N_EVAL * .7))) + rng.sample(test_cb, min(len(test_cb), int(N_EVAL * .3)))
eval_items = [{"question": e["question"], "reference": re.sub(r"\s?\[\d\]", "", e["reference"]),
               "kind": e["kind"], "doc_id": e["doc_id"]} for e in eval_items]
if GOLD_FILE.exists():
    eval_items += [{"question": g["question"], "reference": g["reference_answer"], "kind": "gold",
                    "doc_id": g.get("doc_id", "")} for g in read_jsonl(GOLD_FILE)]
print(f"Itens de avaliação: {len(eval_items)}", dict(Counter(e["kind"] for e in eval_items)))

CONFIGS = {"A: base": (False, False), "B: base+RAG": (True, False), "C: FT": (False, True), "D: FT+RAG": (True, True)}
rows = []
for i, e in enumerate(eval_items, 1):
    ctx = hybrid_search(e["question"])
    for name, (use_rag, use_ad) in CONFIGS.items():
        msgs = build_messages(e["question"], ctx if use_rag else None)
        t0 = time.time()
        if use_ad:
            ans = generate(msgs, temperature=0)
        else:
            with model.disable_adapter():
                ans = generate(msgs, temperature=0)
        lat = time.time() - t0
        j = judge(e["question"], e["reference"], ans, ctx if use_rag else None) or {}
        cites = [int(x) for x in re.findall(r"\[(\d)\]", ans)]
        cite_ok = None
        if use_rag and e["kind"] == "rag":
            cite_ok = bool(cites) and any(1 <= n <= len(ctx) and ctx[n - 1]["doc_id"] == e["doc_id"] for n in cites)
        rows.append({"config": name, "kind": e["kind"], "question": e["question"], "answer": ans,
                     "correcao": j.get("correcao"), "fidelidade": j.get("fidelidade"),
                     "cite_ok": cite_ok, "ptbr": is_ptbr(ans), "latency_s": lat})
    if i % 10 == 0:
        print(f"  {i}/{len(eval_items)}")

df_eval = pd.DataFrame(rows)
df_eval.to_csv(OUT_DIR / RUN_NAME / "eval_detalhado.csv", index=False)

In [ ]:
abst = []
for e in rng.sample(test_un, min(30, len(test_un))) + rng.sample(test_rag, min(30, len(test_rag))):
    ctx = [chunk_by_id[c] for c in e["context_chunk_ids"]]
    msgs = build_messages(e["question"], ctx)
    for name, use_ad in {"B: base+RAG": False, "D: FT+RAG": True}.items():
        if use_ad:
            ans = generate(msgs, temperature=0)
        else:
            with model.disable_adapter():
                ans = generate(msgs, temperature=0)
        abst.append({"config": name, "kind": e["kind"], "abstained": is_abstention(ans)})
df_abst = pd.DataFrame(abst)

summary = df_eval.groupby("config").agg(
    correcao=("correcao", "mean"), fidelidade=("fidelidade", "mean"),
    citacao_correta=("cite_ok", lambda s: s.dropna().astype(float).mean() if s.notna().any() else np.nan),
    ptbr=("ptbr", "mean"), latencia_s=("latency_s", "mean")).round(3)
if len(df_abst):
    summary["abstencao_correta"] = df_abst[df_abst.kind == "unanswerable"].groupby("config").abstained.mean()
    summary["falsa_abstencao"] = df_abst[df_abst.kind == "rag"].groupby("config").abstained.mean()
summary.to_csv(OUT_DIR / RUN_NAME / "eval_resumo.csv")
json.dump({"retrieval": retrieval_report, "summary": json.loads(summary.reset_index().to_json(orient="records"))},
          open(OUT_DIR / RUN_NAME / "eval_resumo.json", "w"), indent=2, ensure_ascii=False)
display(summary)

ax = summary[["correcao", "fidelidade"]].plot.bar(figsize=(8, 4), rot=0, ylim=(0, 5))
ax.set_title("Gemma 4 E2B · Refauna — notas do juiz (1–5)"); ax.grid(axis="y", alpha=.3)
plt.savefig(OUT_DIR / RUN_NAME / "eval_notas.png", dpi=160, bbox_inches="tight"); plt.show()

## 8. Exportação e publicação no Hugging Face

| Repositório | Conteúdo | Para quê |
|---|---|---|
| `gemma-4-e2b-refauna-lora` | adapter LoRA (dezenas de MB) | reprodutibilidade, combinar com o base |
| `gemma-4-e2b-refauna` | modelo mesclado 16-bit | Transformers / vLLM |
| `gemma-4-e2b-refauna-GGUF` | Q4_K_M e Q8_0 | **Ollama / llama.cpp no Mac M1** e no Space |
| `refauna-sft-ptbr` | dataset de treino (privado) | transparência |
| `refauna-rag-chunks` | chunks + embeddings (só documentos redistribuíveis) | RAG no Space |

Tudo é criado **privado** (`HF_PRIVATE = True`). Revise os model cards e torne público pelo site quando quiser.

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
for repo in (REPO_LORA, REPO_MERGED, REPO_GGUF):
    api.create_repo(repo, private=HF_PRIVATE, exist_ok=True)

model.push_to_hub(REPO_LORA, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_LORA, token=HF_TOKEN)
print("✅ LoRA:", f"https://huggingface.co/{REPO_LORA}")

In [ ]:
model.push_to_hub_merged(REPO_MERGED, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
print("✅ Merged 16-bit:", f"https://huggingface.co/{REPO_MERGED}")

In [ ]:
# Converte com llama.cpp e envia os GGUF (leva alguns minutos)
model.push_to_hub_gguf(REPO_GGUF, tokenizer, quantization_method=["q4_k_m", "q8_0"], token=HF_TOKEN)
print("✅ GGUF:", f"https://huggingface.co/{REPO_GGUF}")

In [ ]:
def eval_table_md():
    try:
        s = json.load(open(OUT_DIR / RUN_NAME / "eval_resumo.json"))
    except FileNotFoundError:
        return "_Avaliação ainda não executada._"
    cols = ["config", "correcao", "fidelidade", "citacao_correta", "abstencao_correta", "ptbr"]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in s["summary"]:
        lines.append("| " + " | ".join("" if r.get(c) is None else (f"{r[c]:.2f}" if isinstance(r[c], float) else str(r[c])) for c in cols) + " |")
    rr = s["retrieval"]
    lines.append(f"\nRecuperação híbrida (teste, n={rr['n']}): Recall@{TOP_K} = {rr[f'recall@{TOP_K}']:.2f}, MRR@20 = {rr['mrr@20']:.2f}")
    return "\n".join(lines)

def model_card(library):
    refs = "\n".join(f"- {d['title']} ({d.get('year') or 's.d.'})" + (f". DOI: [{d['doi']}](https://doi.org/{d['doi']})" if d.get("doi") else "")
                     for d in sorted(docs, key=lambda d: (d.get("year") or 0)))
    return f"""---
license: apache-2.0
base_model: google/gemma-4-e2b-it
language: [pt, en]
library_name: {library}
pipeline_tag: text-generation
tags: [gemma4, unsloth, qlora, lora, rag, refauna, biodiversidade, conservacao, mata-atlantica, refaunacao]
---

# Gemma 4 E2B · Refauna (PT-BR)

Small Language Model especializado em **defaunação, refaunação, reintrodução de fauna, dispersão de sementes e conservação da Mata Atlântica**, a partir da produção científica do [Refauna](https://refauna.org.br) — iniciativa criada em 2010 para devolver animais silvestres nativos aos seus habitats na Mata Atlântica do Rio de Janeiro.

Desenvolvido para a palestra *"Pequenos modelos, conhecimento especializado: fine-tuning e RAG com os dados do Refauna"* (TDC Stage @ Rec'n'Play 2026).

**English:** Gemma 4 E2B fine-tuned (QLoRA, Unsloth) in Brazilian Portuguese to answer questions about Atlantic Forest rewilding, grounded on retrieved passages from Refauna research papers with [n] citations.

## Como foi treinado
- **Base:** `google/gemma-4-e2b-it` (Apache 2.0) via `{BASE_MODEL}`
- **Método:** {"QLoRA 4-bit" if LOAD_IN_4BIT else "LoRA 16-bit"}, r={LORA_R}, alpha={LORA_R}, {EPOCHS} épocas, lr={LR}, só camadas de linguagem, template `gemma-4` (sem thinking)
- **Dados:** {len(train)} exemplos sintéticos em PT-BR gerados a partir de {len(docs)} artigos (estilo RAFT: ~60% com contexto + citações, ~15% abstenção, ~25% sem contexto), com revisão humana amostral e split por documento
- **Hardware:** {train_report['gpu']}, {train_report['minutes']} min, pico de {train_report['peak_vram_gb']} GB de VRAM

## Uso recomendado: com RAG
O modelo foi treinado para receber trechos numerados e citar `[n]`. Formato do prompt:

```
system: {SYSTEM_PROMPT}
user:   CONTEXTO:
        [1] (Autor et al., ano — Título) trecho...
        [2] ...
        PERGUNTA: ...
```

Embeddings usados no RAG: `google/embeddinggemma-300m` + BM25 (fusão RRF), top-{TOP_K}.

### Ollama (Mac/Linux/Windows)
```bash
ollama run hf.co/{REPO_GGUF}:Q4_K_M
```

### Transformers
```python
from transformers import pipeline
pipe = pipeline("text-generation", model="{REPO_MERGED}")
```

## Avaliação
{eval_table_md()}

Notas de 1 a 5 atribuídas por um juiz LLM; A = base, B = base+RAG, C = fine-tunado, D = fine-tunado+RAG.

## Limitações
- Uso educacional e de divulgação científica; **não substitui a literatura original** — sempre consulte as fontes citadas.
- Sem RAG, o modelo pode errar números, datas e nomes de espécies.
- Conhecimento restrito aos artigos usados; não cobre todo o trabalho do Refauna.

## Fontes (atribuição)
{refs}
"""

for repo, lib in [(REPO_LORA, "peft"), (REPO_MERGED, "transformers"), (REPO_GGUF, "gguf")]:
    api.upload_file(path_or_fileobj=model_card(lib).encode("utf-8"), path_in_repo="README.md",
                    repo_id=repo, commit_message="Model card")
print("✅ Model cards enviados")

In [ ]:
from datasets import DatasetDict
DatasetDict({s: Dataset.from_list(r) for s, r in {"train": train, "validation": val, "test": test}.items()}
            ).push_to_hub(REPO_SFT, private=True, token=HF_TOKEN)
print("✅ Dataset SFT (privado):", f"https://huggingface.co/datasets/{REPO_SFT}")

pub = [{**c, "embedding": doc_emb[i].tolist()} for i, c in enumerate(chunks) if c.get("redistributable")]
if pub:
    Dataset.from_list(pub).push_to_hub(REPO_CHUNKS, private=HF_PRIVATE, token=HF_TOKEN)
    print(f"✅ {len(pub)} chunks redistribuíveis:", f"https://huggingface.co/datasets/{REPO_CHUNKS}")
else:
    print("Nenhum documento marcado como redistributable=true no sources.csv — chunks não publicados.")

### (Opcional) QLoRA vs LoRA 16-bit
Rode o notebook com `LOAD_IN_4BIT = True`, depois **reinicie a sessão**, mude para `False` e rode de novo a partir da seção 6 (a geração sintética fica em cache e não é refeita; recarregue as seções 0–5 normalmente). A célula abaixo compara os runs salvos no Drive.

In [ ]:
runs = []
for p in sorted(OUT_DIR.glob("*/train_report.json")):
    r = json.load(open(p))
    ev = p.parent / "eval_resumo.csv"
    if ev.exists():
        d = pd.read_csv(ev).set_index("config")
        if "D: FT+RAG" in d.index:
            r["D_correcao"] = d.loc["D: FT+RAG", "correcao"]; r["D_fidelidade"] = d.loc["D: FT+RAG", "fidelidade"]
    runs.append(r)
pd.DataFrame(runs)

## 9. Levando para o MacBook Air M1 (8 GB)

**Modelo (Ollama):**
```bash
brew install ollama            # ou baixe o app em ollama.com
ollama pull gemma4:e2b         # modelo base, para a comparação ao vivo
ollama run hf.co/SEU_USUARIO_HF/gemma-4-e2b-refauna-GGUF:Q4_K_M
```
> Enquanto o repositório for **privado**, o Ollama precisa de acesso: adicione a chave pública do Ollama (`cat ~/.ollama/id_ed25519.pub`) em *Settings → SSH keys* no Hugging Face, ou baixe o arquivo com `hf download` e use `FROM ./arquivo.gguf` num Modelfile.

**Índice RAG:** copie `refauna-slm/data/chunks.jsonl` para `data/` do repositório e **reconstrua o índice no Mac** com a CLI do projeto (`refauna index`). É mais robusto do que copiar a pasta do Chroma (versões diferentes do chromadb podem ser incompatíveis) e leva poucos minutos no M1.

**Memória no M1 8 GB:** GGUF Q4_K_M (~3 GB) + EmbeddingGemma em float32 (~1,2 GB) cabem com folga se você fechar o navegador pesado durante a demo. Use `num_ctx 4096`.

**Teste de paridade:** rode as mesmas 10 perguntas aqui (`ask`) e no Ollama. Respostas muito diferentes em formato = template de chat divergente no GGUF.